# MedGemma 4B — Modelo base (Colab)

MedGemma 4B **sin fine-tuning**. Toda la logica vive en el paquete `medgemma_glaucoma/`; este notebook solo lo orquesta.

Dos modos:
- **`classifier`** — glaucoma / no glaucoma (dataset local).
- **`descriptor`** — describe la imagen (descripciones de Hugging Face; cambiable).

## 1. Instalar dependencias

In [ ]:
!pip install -q -U transformers accelerate timm hf_transfer datasets

## 2. Cargar el paquete (local o clonando el repo en Colab)

In [ ]:
# Deja el paquete `medgemma_glaucoma` importable.
#  - Local: ya esta en el repo, no hace nada.
#  - Colab: clona el repo (ajusta REPO_URL) y entra en su carpeta.
import os, sys, importlib.util

REPO_URL = "https://github.com/Luco1421/utils_medgemma.git"  # <-- ajusta a tu repo
REPO_DIR = "utils_medgemma"

if importlib.util.find_spec("medgemma_glaucoma") is None:
    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone -q {REPO_URL}")
    if os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)
    sys.path.insert(0, os.getcwd())

import medgemma_glaucoma as mg
print("paquete OK | cwd:", os.getcwd())

## 3. Autenticación en Hugging Face (modelo gated)

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # descarga mas rapida

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

from huggingface_hub import login
login(token=token) if token else login()

## 4. Cargar modelo y processor

In [ ]:
model, processor = mg.load_model()

## 5. Función de chat (prueba unitaria)

Un turno imagen + pregunta para validar el modelo a mano.

In [ ]:
from PIL import Image

image = Image.open("glaucoma.jpg").convert("RGB")  # sube una imagen de prueba
print(mg.ask_image(model, processor,
                   image,
                   "Describe esta imagen de fondo de ojo y dime si hay senales de glaucoma"))

## 6. Evaluación zero-shot del modelo base

In [ ]:
MODE      = "classifier"   # "classifier" | "descriptor"
N_SAMPLES = 12

records = mg.load_records(mode=MODE, limit=300)
print(f"Modo={MODE} | ejemplos disponibles: {len(records)}")

if MODE == "classifier":
    _, sample = mg.train_test_split(records, n_test=N_SAMPLES)
else:
    sample = records[:N_SAMPLES]

mg.evaluate(model, processor, sample, mode=MODE)